# Nemotron 3.5 Embed 1B — visual RAG on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Nemotron-3.5-Embed-1B-visual-rag.ipynb)

[`nvidia/Nemotron-3.5-Embed-1B-BF16`](https://huggingface.co/nvidia/Nemotron-3.5-Embed-1B-BF16) is NVIDIA's 1.55B multimodal embedding model (OpenMDW-1.1). It maps a **page image**, a **text passage**, or **image + text** into one 2048-dimensional L2-normalized vector. Queries are text only. That is the difference from the [EmbeddingGemma 2 notebook](embeddinggemma-2-inference-similarity.ipynb) in this repo, which stays text-only. This notebook builds a tiny visual RAG loop: embed a few arXiv page images plus text snippets, index them in FAISS, retrieve ranked hits, then answer from the retrieved context with the same small companion LLM already used in this repo (`Qwen/Qwen2.5-1.5B-Instruct`). The embedder is the focus; the reader is a short last step.

The language tower shares an embedding space with [`nvidia/Nemotron-3-Embed-1B-BF16`](https://huggingface.co/nvidia/Nemotron-3-Embed-1B-BF16). The vision encoder is a Pixtral stack initialized from Ministral-3. Mean pooling + L2 norm produces the vector. You can truncate to 1024 or 512 dims and re-normalize; this notebook keeps the full 2048.

**Licence:** [OpenMDW-1.1](https://openmdw.ai/license/1-1/). The Ministral-3 base models are Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has no native bfloat16 and no FlashAttention 2. Official examples load **bfloat16 + FlashAttention 2** on Ampere / Hopper / Blackwell. This notebook loads **float16 + SDPA** instead. The Hub file is about 3.1 GB of BF16 weights; on the T4 the tower sits around **3.5 GB** in float16. Encoding a page whose longest edge is the card default (1,120 px) should stay well inside 16 GB. Scores will not match the card's printed BF16 decimals.

**Links:** [model card](https://huggingface.co/nvidia/Nemotron-3.5-Embed-1B-BF16) · [blog](https://huggingface.co/blog/nvidia/nemotron-retriever-world-class-agentic-retrieval)


In [ ]:
%pip install -q "sentence-transformers[image]==6.1.0" "transformers==5.20.1" pillow pypdfium2==5.14.0 faiss-cpu

# The model card uses sentence-transformers[image]>5.4 and transformers==5.20.0.
# 6.1.0 / 5.20.1 match the other T4 notebooks in this repo and sit in that range.
# pypdfium2 renders PDF pages. faiss-cpu is enough for a handful of 2048-d vectors.
# Do not install flash-attn: a T4 cannot run FlashAttention 2.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


In [ ]:
import torch
import transformers
from packaging.version import Version
import sentence_transformers

assert Version(transformers.__version__) >= Version("5.20.0"), (
    f"transformers {transformers.__version__} is loaded; Nemotron-3.5-Embed needs 5.20.0+. "
    "Run the install cell, then Runtime > Restart session."
)
assert Version(sentence_transformers.__version__) >= Version("5.4.0")

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2 and no native BF16 compute.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("sentence-transformers", sentence_transformers.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


## 1. Load the embedder

`SentenceTransformer` is the path on the model card. Saved prompts are `query: ` and `passage: `; `encode_query` / `encode_document` apply them. `model_kwargs` forwards **float16 + SDPA** on a T4 instead of the checkpoint's bfloat16 + FlashAttention 2 default.


In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_ID = "nvidia/Nemotron-3.5-Embed-1B-BF16"


def load_embedder(attn):
    return SentenceTransformer(
        MODEL_ID,
        device="cuda",
        model_kwargs={"dtype": DTYPE, "attn_implementation": attn},
    )


try:
    embedder = load_embedder("sdpa")
    print("attention: sdpa")
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    embedder = load_embedder("eager")
    print("attention: eager")

print("parameters:", f"{sum(p.numel() for p in embedder.parameters()) / 1e9:.2f}B")
print("dtype:", embedder.dtype)
print("embedding dim:", embedder.get_embedding_dimension())
print("prompts:", getattr(embedder, "prompts", None))
print("similarity:", getattr(embedder, "similarity_fn_name", None))
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


## 2. Embed a few PDF pages and text snippets

Each visual document is a public arXiv first page (or the Transformer architecture figure on page 3), rendered and capped at **1,120 px** — the card's default `longest_edge`. Documents can be an image, text, or both. Queries stay text-only. `batch_size=1` keeps peak VRAM down while encoding pages.


In [ ]:
import pypdfium2 as pdfium
import requests

UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}
ATTENTION = "https://arxiv.org/pdf/1706.03762"
BERT = "https://arxiv.org/pdf/1810.04805"
GPT3 = "https://arxiv.org/pdf/2005.14165"


def load_pdf(url):
    response = requests.get(url, headers=UA, timeout=120)
    response.raise_for_status()
    return pdfium.PdfDocument(response.content)


def render_page(pdf, page_index=0, longest=1120):
    image = pdf[page_index].render(scale=150 / 72).to_pil().convert("RGB")
    image.thumbnail((longest, longest))
    return image


attention_pdf = load_pdf(ATTENTION)
bert_pdf = load_pdf(BERT)
gpt3_pdf = load_pdf(GPT3)
attention_p1 = render_page(attention_pdf, 0)
attention_fig = render_page(attention_pdf, 2)
bert_p1 = render_page(bert_pdf, 0)
gpt3_p1 = render_page(gpt3_pdf, 0)

print(f"Attention p1: {attention_p1.size}")
print(f"Attention fig (p3): {attention_fig.size}")
print(f"BERT p1: {bert_p1.size}")
print(f"GPT-3 p1: {gpt3_p1.size}")


In [ ]:
ATTENTION_TEXT = (
    "The Transformer is a sequence transduction model based solely on attention, "
    "replacing recurrence and convolutions. Multi-head self-attention and scaled "
    "dot-product attention let it model global dependencies."
)
BERT_TEXT = (
    "BERT is a bidirectional Transformer pretrained with a masked language model "
    "and next-sentence prediction, then fine-tuned on GLUE and SQuAD."
)
GPT3_TEXT = (
    "GPT-3 is a 175 billion parameter autoregressive language model that performs "
    "few-shot, one-shot, and zero-shot tasks via in-context learning without gradient updates."
)
DISTRACTOR_TEXT = (
    "A biological foundation model designed to analyze and generate DNA, RNA, and protein sequences."
)

# Stored `text` is what the reader LLM sees after retrieval. The `payload` is what
# the embedder encodes: image, text, or both, which is the point of this model.
CORPUS = [
    {
        "title": "Attention Is All You Need — page 1 (image + text)",
        "kind": "image+text",
        "image": attention_p1,
        "text": ATTENTION_TEXT,
        "payload": {"image": attention_p1, "text": ATTENTION_TEXT},
    },
    {
        "title": "Attention Is All You Need — architecture figure (image)",
        "kind": "image",
        "image": attention_fig,
        "text": (
            "Figure 1 of the Transformer paper: encoder-decoder stacks with multi-head "
            "attention and position-wise feed-forward layers."
        ),
        "payload": {"image": attention_fig},
    },
    {
        "title": "BERT — page 1 (image + text)",
        "kind": "image+text",
        "image": bert_p1,
        "text": BERT_TEXT,
        "payload": {"image": bert_p1, "text": BERT_TEXT},
    },
    {
        "title": "GPT-3 — page 1 (image)",
        "kind": "image",
        "image": gpt3_p1,
        "text": GPT3_TEXT,
        "payload": {"image": gpt3_p1},
    },
    {
        "title": "Biology foundation model (text only)",
        "kind": "text",
        "image": None,
        "text": DISTRACTOR_TEXT,
        "payload": DISTRACTOR_TEXT,
    },
]


def embed_one(payload):
    encoded = embedder.encode_document([payload], batch_size=1)
    vec = encoded[0] if isinstance(encoded, list) else encoded
    if getattr(vec, "ndim", 1) == 2:
        vec = vec[0]
    return vec


doc_vectors = [embed_one(row["payload"]) for row in CORPUS]
print("documents:", len(doc_vectors), "| dim:", len(doc_vectors[0]))
print("first 5 values:", [round(float(x), 4) for x in doc_vectors[0][:5]])
print("L2 norm:", round(float((doc_vectors[0] ** 2).sum() ** 0.5), 4))
print(f"VRAM allocated after encode: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


## 3. Tiny FAISS index, then rank hits

Embeddings are already L2-normalized, so inner product is cosine similarity. `IndexFlatIP` is enough for five vectors. Each query is encoded with `encode_query` (`query: ` prefix).


In [ ]:
import numpy as np
import faiss
import matplotlib.pyplot as plt

def as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().float().cpu().numpy()
    return np.ascontiguousarray(x, dtype=np.float32)


doc_matrix = as_numpy(doc_vectors)
if doc_matrix.ndim == 1:
    doc_matrix = doc_matrix.reshape(1, -1)
index = faiss.IndexFlatIP(int(doc_matrix.shape[1]))
index.add(doc_matrix)
print("FAISS ntotal:", index.ntotal, "| dim:", index.d)

QUERIES = [
    "How does scaled dot-product attention work in the Transformer?",
    "encoder-decoder architecture diagram with multi-head attention blocks",
    "bidirectional transformers pretrained with a masked language model",
    "175 billion parameter few-shot learners and in-context learning",
]

query_matrix = as_numpy(embedder.encode_query(QUERIES))
if query_matrix.ndim == 1:
    query_matrix = query_matrix.reshape(1, -1)
scores, hits = index.search(query_matrix, k=min(3, index.ntotal))

for query, row_scores, row_hits in zip(QUERIES, scores, hits):
    print(f"\nQ: {query}")
    for rank, (idx, score) in enumerate(zip(row_hits, row_scores), start=1):
        print(f"  {rank}. {score:.4f}  [{CORPUS[int(idx)]['kind']}] {CORPUS[int(idx)]['title']}")

# Show the first query's ranking over the visual pages.
visual = [(i, row) for i, row in enumerate(CORPUS) if row["image"] is not None]
fig, axes = plt.subplots(1, len(visual), figsize=(14, 4))
first_scores = {int(i): float(s) for i, s in zip(hits[0], scores[0])}
best = int(hits[0][0])
for ax, (i, row) in zip(axes, visual):
    ax.imshow(row["image"])
    ax.set_axis_off()
    mark = "  << best" if i == best else ""
    shown = first_scores.get(i)
    label = f"{shown:.2f}{mark}" if shown is not None else row["kind"]
    ax.set_title(f"{label}\n{row['title']}", fontsize=7)
fig.suptitle("Text query → page image / image+text (cosine)", fontsize=13)
fig.tight_layout()
plt.show()


## 4. Short answer from retrieved context

The embedder only retrieves. A small companion LLM already used in this repo, [`Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), writes a one-paragraph answer from the top hits. The embedder is freed first so the T4 is not holding both towers. Swap the reader for any other small instruct model if you want; keep the index.


In [ ]:
import gc

from transformers import AutoModelForCausalLM, AutoTokenizer

QUESTION = QUERIES[0]
top_k = 3
top_idxs = [int(i) for i in hits[0][:top_k]]
context = "\n\n".join(
    f"[{CORPUS[i]['title']}]\n{CORPUS[i]['text']}" for i in top_idxs
)
print("question:", QUESTION)
print("context titles:")
for i in top_idxs:
    print(f"  - {CORPUS[i]['title']}")

del embedder
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after freeing embedder: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

READER_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(READER_ID)
reader = AutoModelForCausalLM.from_pretrained(READER_ID, dtype=DTYPE).eval().to("cuda")
print(f"VRAM with reader: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

messages = [
    {
        "role": "system",
        "content": (
            "Answer the question using only the retrieved passages. "
            "Be concise. If the passages do not contain the answer, say you do not know."
        ),
    },
    {"role": "user", "content": f"Passages:\n{context}\n\nQuestion: {QUESTION}"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(reader.device)
with torch.inference_mode():
    out = reader.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
answer = tokenizer.decode(out[0, inputs["input_ids"].shape[-1] :], skip_special_tokens=True)
print("\nanswer:\n", answer.strip())


## Licence and caveats

- **Weights:** [`nvidia/Nemotron-3.5-Embed-1B-BF16`](https://huggingface.co/nvidia/Nemotron-3.5-Embed-1B-BF16) is [OpenMDW-1.1](https://openmdw.ai/license/1-1/). The Ministral-3 vision/language bases are Apache 2.0. The companion reader is [`Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) (Apache 2.0).
- **Not the official runtime.** The card's examples use bfloat16 and FlashAttention 2 on Ampere / Hopper / Blackwell. A free T4 has neither, so this notebook uses float16 and SDPA. Retrieval scores can drift from the printed BF16 numbers.
- **Hardware.** NVIDIA lists Ampere, Hopper, and Blackwell. Turing (T4) is a Colab convenience path, not a supported microarchitecture on the card.
- **Retrieval is not guaranteed.** The model does not always return the correct page. Treat ranks as a demo, not a production eval (see ViDoRe V3 / MIRACL-VISION on the card).
- **Inputs.** Queries are text. Documents may be an image, text, or both. Default image `longest_edge` is 1,120 px (max 1,540). Context limit is 32,768 tokens including image tokens.
- **Rights.** The sample pages are public arXiv PDFs. Do not embed images or documents you do not have rights to use.
- **Scope.** This notebook is about the embedder. The Qwen reader is a short answer-from-context step, the same ungated 1.5B instruct checkpoint used in the [Transformers runtime tasks](AIBackends-Transformers-runtime-tasks.ipynb) notebook. For text-only embeddings, use [EmbeddingGemma 2](embeddinggemma-2-inference-similarity.ipynb). For late-interaction page search without a pooled vector, use [pplx-embed-v2](pplx-embed-v2-late-0.6b-page-search.ipynb).
